# BRONZE_LEITOS

**Origem:** arquivos `LTAL{AA}{MM}.dbc` do CNES (leitos por estabelecimento de Alagoas), um por mês:
`ftp://ftp.datasus.gov.br/dissemin/publicos/CNES/200508_/Dados/LT/`

Cada linha é um tipo de leito (`CODLEITO`) de um estabelecimento na competência, com a
quantidade existente (`QT_EXIST`) e a disponível ao SUS (`QT_SUS`).

**Idempotência:** a tabela é particionada por ano. Antes de carregar um ano, a partição dele é truncada (`TRUNCATE PARTITION FOR`), então reprocessar nunca duplica dados.

**Falhas:** se algum arquivo não puder ser baixado ou lido, o notebook para com erro e a task do Airflow fica vermelha.

Anos processados: variável de ambiente `ANOS_PIPELINE` (padrão `2023,2024`).

In [ ]:
import sys
sys.path.append("..")
from dw_utils import carregar_arquivos_mensais, consultar

In [ ]:
COLUNAS = ['COMPETEN', 'CNES', 'CODUFMUN', 'TP_LEITO', 'CODLEITO', 'QT_EXIST', 'QT_SUS']

carregar_arquivos_mensais("BRONZE_LEITOS", "/dissemin/publicos/CNES/200508_/Dados/LT", "LTAL", COLUNAS)

In [ ]:
consultar("""SELECT COMPETEN, COUNT(*) AS LINHAS, SUM(QT_SUS) AS LEITOS_SUS FROM BRONZE_LEITOS GROUP BY COMPETEN ORDER BY 1""")